# M3 Cross-Encoder A100 Run-All

Canonical full-scoring notebook for Hybrid top-100 union LambdaMART top-50 CE pairs. Runtime: Colab CUDA, preferably A100. Inputs live on Google Drive; checkpoint blocks and final scores are written back to Drive.

Pinned scoring code: `eb67d8f` (`eb67d8f1d8bbba14a58e9a0a12fd787b5efaa01d`). Drive-staged union/archive artifacts were built at `4f327ff` (`4f327ff86c5a50b11e850620e8b2f8d74311721c`).


## 0. Canonical constants

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time

OWNER = 'YashDThapliyal'
REPO = 'adaptirank'
COMMIT = 'eb67d8f1d8bbba14a58e9a0a12fd787b5efaa01d'
ARTIFACT_BASE_COMMIT = '4f327ff86c5a50b11e850620e8b2f8d74311721c'
FP = 'dda38161938e829f2c2fc9b73d40d6cf922a5470c3b45bf176f742ee0ca7c667'
EXPECTED_ROWS = 3156056
EXPECTED_UNION_SHA = '16a43b01f0ba159e5950c1fe7d4363b6c05d7b0c9ffe6c581272379ef9c8488d'
EXPECTED_ARCHIVE_SHA = 'a79bb8ad98b2cdbfb56b6f6680c95ce87ef1dd792a16ac91d95fec563ee67f5f'
DRIVE_ROOT = Path('/content/drive/MyDrive/adaptirank/m3_ce_run')
DATASET_TAR = Path('/content/drive/MyDrive/adaptirank/adaptirank_dataset.tar')
INPUT_ARCHIVE = Path('/content/drive/MyDrive/adaptirank/m3_ce_a100_input.tar.gz')
WORKDIR = Path('/content/adaptirank')
print({'repo': f'{OWNER}/{REPO}', 'commit': COMMIT, 'artifact_base_commit': ARTIFACT_BASE_COMMIT, 'drive_root': str(DRIVE_ROOT)})


## 1. GPU verify

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
try:
    import torch
    print('pre-install torch', torch.__version__, 'cuda', torch.cuda.is_available())
except Exception as exc:
    print('torch precheck deferred:', repr(exc))

## 2. Mount Drive and create directories

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
for path in [DRIVE_ROOT, DRIVE_ROOT / 'checkpoints', DRIVE_ROOT / 'final', DRIVE_ROOT / 'metadata']:
    path.mkdir(parents=True, exist_ok=True)
assert DATASET_TAR.is_file(), DATASET_TAR
assert INPUT_ARCHIVE.is_file(), INPUT_ARCHIVE
print('drive directories ready under', DRIVE_ROOT)

## 3. Clone repo and verify commit

In [ ]:
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
subprocess.run(['git', 'clone', f'https://github.com/{OWNER}/{REPO}.git', str(WORKDIR)], check=True)
os.chdir(WORKDIR)
subprocess.run(['git', 'checkout', COMMIT], check=True)
head = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
dirty = subprocess.run(['git', 'status', '--porcelain'], capture_output=True, text=True, check=True).stdout.strip()
assert head == COMMIT, head
assert dirty == '', dirty
print('checked out clean commit', head)

## 4. Install from lockfile and record environment

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--frozen', '--dev'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-e', '.'], check=True)
env = subprocess.run(['uv', 'pip', 'freeze', '--system'], capture_output=True, text=True, check=True).stdout
(DRIVE_ROOT / 'metadata' / 'environment.txt').write_text(env)
print(env.splitlines()[:10])

## 5. Locate input archive and verify SHA

In [ ]:
sys.path.insert(0, str(WORKDIR / 'src'))
import adaptirank.ranking.ce_workflow as ce_workflow

assert ce_workflow.CANONICAL_GIT_COMMIT == COMMIT
assert ce_workflow.ARTIFACT_BASE_GIT_COMMIT == ARTIFACT_BASE_COMMIT
assert ce_workflow.CANONICAL_DATASET_FINGERPRINT == FP
assert ce_workflow.EXPECTED_CE_UNION_ROWS == EXPECTED_ROWS
assert ce_workflow.EXPECTED_CE_UNION_SHA256 == EXPECTED_UNION_SHA
assert ce_workflow.EXPECTED_ARCHIVE_SHA256 == EXPECTED_ARCHIVE_SHA
archive_sha = ce_workflow.verify_file_sha256(INPUT_ARCHIVE, EXPECTED_ARCHIVE_SHA)
print({'archive': str(INPUT_ARCHIVE), 'sha256': archive_sha})


## 6. Extract archive and dataset, then verify union invariants

In [ ]:
subprocess.run(['tar', '-xzf', str(INPUT_ARCHIVE), '-C', str(WORKDIR)], check=True)
dataset_parent = WORKDIR / 'artifacts/datasets/esci/processed'
dataset_parent.mkdir(parents=True, exist_ok=True)
subprocess.run(['tar', '-xf', str(DATASET_TAR), '-C', str(dataset_parent)], check=True)
union_path = WORKDIR / f'artifacts/ranking/{FP}/m3_three_split/cross_encoder/pair_union.parquet'
union_manifest = union_path.with_name('pair_union_manifest.json')
dataset_dir = WORKDIR / f'artifacts/datasets/esci/processed/{FP}'
assert union_path.is_file(), union_path
assert dataset_dir.joinpath('catalog.parquet').is_file(), dataset_dir
union_stats = ce_workflow.verify_ce_union_files(union_path, manifest_path=union_manifest)
print(json.dumps(union_stats, indent=2, default=str))

## 7. Load pinned CE and probe finite scores

In [ ]:
gpu_info = ce_workflow.require_cuda_gpu(warn_non_a100=True)
from adaptirank.ranking.cross_encoder import CrossEncoderScorer

scorer = CrossEncoderScorer(
    model_name=ce_workflow.PINNED_CE_MODEL,
    model_revision=ce_workflow.PINNED_CE_REVISION,
    device='cuda',
    batch_size=256,
    max_length=512,
)
probe = ce_workflow.probe_cross_encoder(scorer)
print(json.dumps({'gpu': gpu_info, 'probe': probe}, indent=2, default=str))

## 8. Benchmark batch sizes on validation subset

In [ ]:
import polars as pl

pairs = pl.read_parquet(union_path)
validation_pairs = pairs.filter(pl.col('split') == 'validation')
bench_subset = ce_workflow.deterministic_benchmark_subset(validation_pairs, n_pairs=4096)
queries = pl.read_parquet(dataset_dir / 'queries.parquet')
catalog = pl.read_parquet(dataset_dir / 'catalog.parquet')
bench = ce_workflow.benchmark_batch_sizes(
    bench_subset,
    queries,
    catalog,
    batch_sizes=(128, 256, 384),
    device='cuda',
)
ce_workflow.atomic_write_json(DRIVE_ROOT / 'metadata' / 'batch_benchmark.json', {'results': bench})
print(json.dumps(bench, indent=2))

## 9. Print config assertions

In [ ]:
from adaptirank.common.config import load_config
from adaptirank.ranking.config import CrossEncoderRunConfig

cfg = load_config(WORKDIR / 'configs/ranking/cross_encoder_union_m3.yaml', CrossEncoderRunConfig)
assert cfg.dataset_fingerprint == FP
assert cfg.pair_artifact.as_posix().endswith('pair_union.parquet')
assert cfg.cross_encoder.model_name == ce_workflow.PINNED_CE_MODEL
assert cfg.cross_encoder.model_revision == ce_workflow.PINNED_CE_REVISION
assert cfg.cross_encoder.device == 'cuda'
print(cfg.model_dump(mode='json'))

## 10. Full scoring with Drive checkpoints

In [ ]:
checkpoint = DRIVE_ROOT / 'checkpoints' / 'm3_ce_union_scores.parquet'
manifest_path = DRIVE_ROOT / 'checkpoints' / 'm3_ce_union_score_manifest.json'
scorer.batch_size = int(cfg.cross_encoder.batch_size)
started = time.time()
scores = ce_workflow.score_union_with_checkpoints(
    pairs,
    queries,
    catalog,
    scorer,
    fields=cfg.cross_encoder.fields,
    checkpoint_path=checkpoint,
    block_queries=cfg.block_queries,
    expected_rows=EXPECTED_ROWS,
    manifest_path=manifest_path,
    metadata={'git_commit': COMMIT, 'dataset_fingerprint': FP, 'union_sha256': EXPECTED_UNION_SHA},
)
elapsed = time.time() - started
print({'rows': scores.height, 'elapsed_seconds': elapsed, 'checkpoint': str(checkpoint)})

## 11. Atomic writes documented

The workflow writes JSON through `atomic_write_json` and scoring checkpoints through parquet temporary files followed by atomic replacement. Interrupted runs resume from completed parquet blocks and the final checkpoint.

## 12. Consolidate to final parquet

In [ ]:
part_dir = checkpoint.with_suffix(checkpoint.suffix + '.parts')
final_scores = DRIVE_ROOT / 'final' / 'm3_ce_union_scores.parquet'
if part_dir.is_dir():
    consolidated = ce_workflow.consolidate_part_blocks(part_dir, final_scores, expected_pairs=pairs)
else:
    shutil.copy2(checkpoint, final_scores)
    consolidated = pl.read_parquet(final_scores)
print({'final_scores': str(final_scores), 'rows': consolidated.height})

## 13. Verify final invariants

In [ ]:
from adaptirank.data.provenance import sha256_file

final_frame = pl.read_parquet(final_scores)
final_stats = ce_workflow.verify_final_scores(
    pairs.select('query_key', 'product_key', 'split'), final_frame
)
final_sha = sha256_file(final_scores)
print(json.dumps({'stats': final_stats, 'sha256': final_sha}, indent=2, default=str))


## 14. Metadata to Drive

In [ ]:
metadata = {
    'repo': f'{OWNER}/{REPO}',
    'git_commit': COMMIT,
    'artifact_base_git_commit': ARTIFACT_BASE_COMMIT,
    'dataset_fingerprint': FP,
    'union_rows': EXPECTED_ROWS,
    'union_sha256': EXPECTED_UNION_SHA,
    'archive_sha256': EXPECTED_ARCHIVE_SHA,
    'checkpoint': str(checkpoint),
    'final_scores': str(final_scores),
    'final_score_stats': final_stats,
    'gpu': gpu_info,
    'benchmark': bench,
    'model_name': ce_workflow.PINNED_CE_MODEL,
    'model_revision': ce_workflow.PINNED_CE_REVISION,
}
ce_workflow.atomic_write_json(DRIVE_ROOT / 'metadata' / 'm3_ce_run_metadata.json', metadata)
print(json.dumps(metadata, indent=2, default=str))


## 15. Verify durable saves

In [ ]:
required = [checkpoint, manifest_path, final_scores, DRIVE_ROOT / 'metadata' / 'm3_ce_run_metadata.json']
for path in required:
    assert path.is_file(), path
    print(path, path.stat().st_size)
ce_workflow.verify_final_scores(pairs.select('query_key', 'product_key', 'split'), pl.read_parquet(final_scores))
print('durable saves verified')

## 16. Optional download bundle

In [ ]:
bundle = DRIVE_ROOT / 'final' / 'm3_ce_a100_outputs.tar.gz'
subprocess.run([
    'tar', '-czf', str(bundle),
    '-C', str(DRIVE_ROOT),
    'final/m3_ce_union_scores.parquet',
    'metadata/m3_ce_run_metadata.json',
    'metadata/batch_benchmark.json',
    'checkpoints/m3_ce_union_score_manifest.json',
], check=True)
print('bundle:', bundle, bundle.stat().st_size)

## 17. Completion report table

In [ ]:
report = pl.DataFrame([
    {'check': 'commit', 'value': COMMIT, 'status': 'PASS'},
    {'check': 'dataset_fingerprint', 'value': FP, 'status': 'PASS'},
    {'check': 'union_rows', 'value': str(EXPECTED_ROWS), 'status': 'PASS'},
    {'check': 'union_sha256', 'value': EXPECTED_UNION_SHA, 'status': 'PASS'},
    {'check': 'final_rows', 'value': str(final_frame.height), 'status': 'PASS'},
    {'check': 'final_scores_path', 'value': str(final_scores), 'status': 'PASS'},
])
print(report)
assert final_frame.height == EXPECTED_ROWS
print('M3 CE A100 run complete')